___

# <font color= #EF4444> **Speech Radicalization Analysis - Report** </font>
#### <font color= #f0565e> `Deep Learning`</font>
<Strong> Sofía Maldonado, Oscar Josué Rocha & Viviana Toledo </Strong>

_12/05/2026._

___

### <font color= #EF4444> **Objective** </font>

This project's objective was to use a transformer to classify a tweet's specific type of radical speech. **Radical speech** refers to any message that entails an extreme form of (in this case) negative emotion towards something, someone or a group of people. Not all radical speech is the same. For the purposes of this project, we identified **six (6)** different types of radical speech:
- ***Hate Speech***: Broader category within speech radicalization. Refers to offensive discourse targeting a group or an individual based on inherent characteristics (such as race, religion or gender) and that may threaten social peace. Hate speech is any kind of communication that attacks or uses pejorative or discriminatory language with reference to a person or group on the basis of who they are.
- ***Dangerous Speech***: Any form of communication that can increase the risk that its audience will condone or commit violence against members of another group. Promotes fear by villainizing another group, such that violence against them seems defensive and justifiable. It uses tactics of dehumanization, mirror accusations (the party which is using terror will accuse the enemy of using terror), threats to group integrity or purity, assertions of attack against women or children, and it questions group loyalty if the speech is not being enacted by the very members of the group.
- ***Extreme Speech***: Hostilities that build structures of exclusion and prepare for violence. Focuses on normalizing violence, exclusion and hostility via humor. Frivolizes violent and exclusionary actions. Emphasizes ""us vs them"" ideology.
- ***Enraged Speech***: Exaltation, both positive and negative, of qualities and actions of a political figure or leader. Polarizes public conversation and emphasizes absolutism. Catalyst for other forms of radicalization and exclusion of those who are different.
- ***Hate Grammar***: Hate as a disciplinary and normative force against ""uncivilized"" or ""unregulated"" groups to ""put them back into order"". Hate Grammar expresses a desire to eliminate others, ensuring groups stay in place. It conveys a fantasy to cleanse public space and hide away members of disident communities.
- ***Horror Grammar***: Normalization of death and State cruelty. Series of media strategies that desensitize the public to barbarism. Forms in which systematically pejorative and exclusionary discursive formulations become entrenched or normalized.

These are all different categories for different types of messages, but they can sometimes overlap. Our idea for this project was to be able to use a model that can understand the nuances between the categories and be able to classify a certain text accordingly, where even humans could not tell the difference.

### <font color= #EF4444> **Data** </font>

Our data consists of approximately 6200 tweets. We decided to search for tweets that talked about controversial issues. The tweets we obtained talk about the following issues:
- Asian Hate, especially in relation to the coronavirus pandemic
- ICE, in the context of the agency's incursion into Minneapolis in late 2025 and early 2026
- Islamophobia, specifically european islamophobia due to mass migration
- LGBT prejudice, in general
- Palestine / Israel, in general

(aquí explican como descargaste los tweets porfi)

The tweets themselves weren't classified in our 6 categories when we downloaded them, since we only used web-scraping. Since we had a lot of tweets, manually classifying the data was not really feasible in the given timeframe, so we decided to automate the classification with a Large Language Model. We tested various cutting edge models but ended up using GPT 5.4, by OpenAI. It was given the following prompt:

```
I'm working on a speech radicalization project, that aims to identify categories of hate speech in a tweet dataset. For this, i defined a series of categories with examples. I need your help in classifying around 6k tweet entries, in a CSV, using these categories. The annotated dataset will be used in my task. Use a hierarchy. Hate Speech should have the least priority since it's the broader category. Dangerous Speech focuses on making the other group seem threatening and justify violence, while Extreme Speech focuses on dehumanization and hostility. Also applies to public figures, leaders and commanders. Yes, the difference is that Hate Grammar polices groups and calls for control/removal. While Hate Speech focuses on attacks based on a person's or group's identity. Yes, an important distinction for Horror Grammar is WHO is commiting the harm Take a look at the categories. Understand and read them carefully. If you have any doubts, ask before working on the annotations. Use the category, revised_definition and example columns
```

After giving the model some clarification, it returned with our tweet CSVs, annotated with the radical speech labels. Something relevant to note here is that the model overwhelmingly classified tweets as "Hate Speech" (approximately 3/4ths of the data) despite the instructions given in the initial prompt to have the least priority. This was our biggest challenge when preparating the data, and we weren't able to have the model give us more balanced results (this could, of course, also be a consequence of the data itself). We also considered having the model return "None" when it wasn't fully convinced of its label and for us to manually write it in, but too many texts were being returned without a label for this to be feasible.


For ease, all tweets were merged into a single CSV file, called ```speech_classified.csv```

### <font color= #EF4444> **Modeling** </font>

***Embeddings - BERTweet***

Our first step in modeling was creating the word embeddings that the models would actually use. For this, we decided to use ***BERTweet***, a custom version of BERT specifically designed for english-language tweets (taking into account the informal speech found on the platform), designed by ***VinAI Research*** [1]. (Fun fact: they were acquired by Qualcomm in 2025).

***Baseline - Starter Models***

We knew we were going to use a transformer model to classify the texts later on, but decided to try out more "normal" models as a baseline to compare. We therefore decided to use an XGBoost classifier, since it is probably the most robust classifier  model we could use that wasn't a neural network, and a Logistic Regression model, for the simplest possible approach. We used Optuna to run different experiments with different hyperparameter combinations for XGBoost, changing the model's learning rate, n_estimators, gamma, and other parameters. We optimized for the highest possible F-1 Score.

***Transformer Model***

For our true classifier using a transformer, we used the same ***BERTweet*** model we used to create the embeddings, for full compatibility. We used a learning rate of 2e-5, and trained for 10 epochs.

(les invito a que extiendan más la explicación del modelado aquí plis and thank you)


### <font color= #EF4444> **Results** </font>

***XGBoost***

In the different trials for XGBoost, we got between $0.21$ and $0.28$ in F-1 Score. We ran a total of 31 trials before stopping when we found out we weren't really getting useful results.

***LR***

For Logistic Regression, we used cross validation to get different results for the model. Our mean F-1 Score from the 5-fold cross validation was $0.2969$ 

***BERTweet***

For the transformer, we checked all the main classification metrics (accuracy, Precision, Recall and F1). We got the following results:
- Accuracy: $0.830386$
- Precision: $0.857173$	 
- Recall: $0.830386$
- F1: $0.839299$

These are very decent results, and considerably higher than the ones we obtained using regular classification models. 

![BERTweet Confusion Matrix](figures/bertweet_cm.png)


Looking at the Confusion Matrix generated by the test of our model, we can see that most of the mistakes, unsurprisingly, come from the model not being able to specifically distinguish hate speech and other types of radical speech. Besides this, the model rarely makes mistakes between the other categories, which is pleasantly surprising.

### <font color= #EF4444> **Challenges** </font>

By far, our biggest challenge for this project was the data. As stated above, time constraints forced us to require automatic labeling of the texts we were going to use for the project. GPT 5.4, the model we used, is definitely very powerful but the classification would have been a lot better had it been made by hand, by people that fully understood the definitions and differences of the types of radical speech presented here. It ended up being a lesser issue than we anticipated, since the final product was far better than we could have imagined, but it is an area of improvement.

Another area of improvement is model age. BERTweet was originally released in May 2020. In the 6 years since, the culture of Twitter has changed a lot (especially following the site's adquisition by Elon Musk in 2022). This has changed the way people interact in the platform, and more importantly, it has affected the types of radical speech that are permitted on Twitter. Some of the worst examples were probably omitted from the corpus used when creating BERTweet due to the tighter regulations around radical speech that were present at the time, which don't exist anymore. Creating a new classifier transformer model with more modern data could be an interesting study project in the future that could provide better results for tasks like this one.

### <font color= #EF4444> **Applications** </font>

This model could be used by think tanks to understand the types of conversations that are happening on a specific platform. Of course, Twitter would be the ideal place of study but other text-based social media websites like Reddit or Tumblr could be looked at using this tool.

This can also be a tool for law enforcement. Studies have shown that a rise in radical speech in Twitter has led to a rise in actual hate crimes happening in Spain [2], so understanding where radical spech happens, and who shares it, can be a useful preventive tool to avoid extreme and awful crimes from happening in other parts of the world. 

For our part, we created a small tool that allows a user to demo the model [here](https://huggingface.co/spaces/chofas/bertweet_rad_speech). It only allows for one text at a time, but could be adapted to allow batches of text entries to be processed at once.

### <font color= #EF4444> **Bibliography and References** </font>

[1] VinAI Research. (2020). **BERTweet**. *Hugging Face*. https://huggingface.co/docs/transformers/en/model_doc/bertweet

[2] Amores, J., Blanco-Herrero, D., Sánchez-Holgado, P. & Frías-Vázquez, M. (2021). **Detecting ideological hatred on Twitter. Development and evaluation of a political ideology hate speech detector in tweets in Spanish.** *Scielo*. https://www.scielo.cl/scielo.php?script=sci_arttext&pid=S0719-367X2021000200098